# Assignment 2B — Person 1 (Data + LSTM)

This notebook is **Person 1 only**.

It is based on the files in this folder, especially:
- `Scats Data October 2006.xls` (15‑minute SCATS volumes)
- `COS30019-2025 S1-A2_B.pdf` (Part B requirements)

It implements:
- Load + clean the SCATS 15‑minute flow dataset
- Convert data into sequences `(X, y)`
- Train an **LSTM** model
- Output: trained LSTM + predictions + saved artifacts

## Setup
If you are missing packages, run the install cell once, then restart the kernel.

In [1]:
# Optional: install dependencies (uncomment if needed)
# !pip install -q pandas numpy scikit-learn joblib xlrd openpyxl torch

import os
import math
import json
import random

import numpy as np
import pandas as pd

from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import joblib

import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if device.type == 'cuda':
    torch.cuda.manual_seed_all(SEED)

print('Torch:', torch.__version__)
print('Device:', device)

Torch: 2.10.0+cpu
Device: cpu


## 1) Load dataset (SCATS 15‑minute flows)

In [6]:
# Load dataset (supports .xls/.xlsx or .csv).
# If your tutor/team provides a CSV, set DATA_PATH to that CSV filename.
DATA_PATH = 'Scats Data October 2006.xls'  # or e.g. 'Boroondara_Flows.csv'
SHEET = 'Data'  # only used for Excel

assert os.path.exists(DATA_PATH), f'Missing {DATA_PATH} in the working directory.'

if DATA_PATH.lower().endswith(('.xls', '.xlsx')):
    df_raw = pd.read_excel(DATA_PATH, sheet_name=SHEET, header=1)
elif DATA_PATH.lower().endswith('.csv'):
    df_raw = pd.read_csv(DATA_PATH)
else:
    raise ValueError(f'Unsupported dataset type: {DATA_PATH} (use .xls/.xlsx or .csv)')

print('Loaded:', DATA_PATH)
print('Rows:', len(df_raw), 'Cols:', df_raw.shape[1])
print('Columns:', list(df_raw.columns)[:15], '...')
df_raw.head(3)

Loaded: Scats Data October 2006.xls
Rows: 4192 Cols: 106
Columns: ['SCATS Number', 'Location', 'CD_MELWAY', 'NB_LATITUDE', 'NB_LONGITUDE', 'HF VicRoads Internal', 'VR Internal Stat', 'VR Internal Loc', 'NB_TYPE_SURVEY', 'Date', 'V00', 'V01', 'V02', 'V03', 'V04'] ...


,SCATS Number,Location,CD_MELWAY,NB_LATITUDE,NB_LONGITUDE,HF VicRoads Internal,VR Internal Stat,VR Internal Loc,NB_TYPE_SURVEY,Date,...,V86,V87,V88,V89,V90,V91,V92,V93,V94,V95
0,970,WARRIGAL_RD N of HIGH STREET_RD,060 G10,-37.86703,145.09159,249,182,1,1,2006-10-01 00:15:00,...,114,97,97,66,81,50,59,47,29,34
1,970,WARRIGAL_RD N of HIGH STREET_RD,060 G10,-37.86703,145.09159,249,182,1,1,2006-10-02 00:15:00,...,111,102,107,114,80,60,62,48,44,26
2,970,WARRIGAL_RD N of HIGH STREET_RD,060 G10,-37.86703,145.09159,249,182,1,1,2006-10-03 00:15:00,...,130,132,114,86,93,90,73,57,29,40


## 2) Clean + select flow columns

The SCATS sheet has:
- Site metadata (SCATS Number, lat/lon, etc.)
- `V00` … `V95` = 96 values (15‑minute volumes) per day

We’ll build a time series per row (site‑day), then create sliding windows.

In [7]:
df = df_raw.copy()

# Basic standardization
if 'SCATS Number' in df.columns:
    df['SCATS Number'] = pd.to_numeric(df['SCATS Number'], errors='coerce').astype('Int64')

if 'Date' in df.columns:
    df['Date'] = pd.to_datetime(df['Date'], errors='coerce')

# Identify flow columns V00..V95
flow_cols = [c for c in df.columns if isinstance(c, str) and c.startswith('V') and len(c) == 3]
flow_cols = sorted(flow_cols, key=lambda s: int(s[1:]))

print('Found flow columns:', len(flow_cols), flow_cols[:5], '...', flow_cols[-5:])

# Convert flows to numeric
for c in flow_cols:
    df[c] = pd.to_numeric(df[c], errors='coerce')

# Drop rows missing key fields (if present)
if 'SCATS Number' in df.columns:
    df = df[df['SCATS Number'].notna()].copy()
if 'Date' in df.columns:
    df = df[df['Date'].notna()].copy()

# Remove rows that are entirely empty flows
df = df.dropna(subset=flow_cols, how='all').copy()

# Fill remaining missing flow values with 0 (simplest assumption)
df[flow_cols] = df[flow_cols].fillna(0.0)

# Optional: remove obvious non-data rows
df = df[df[flow_cols].sum(axis=1) > 0].copy()

# Keep lat/lon (if available)
for c in ['NB_LATITUDE', 'NB_LONGITUDE']:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors='coerce')

print('After cleaning:', df.shape)
df[['SCATS Number', 'Date'] + flow_cols[:6]].head(3)

# Export a clean CSV (matches the 'load CSV' requirement for handoff/integration)
os.makedirs('artifacts_person1', exist_ok=True)
cols_for_clean_csv = [c for c in ['SCATS Number', 'Date', 'NB_LATITUDE', 'NB_LONGITUDE'] if c in df.columns] + flow_cols
df[cols_for_clean_csv].to_csv('artifacts_person1/cleaned_flows.csv', index=False)
print('Saved cleaned CSV -> artifacts_person1/cleaned_flows.csv')

Found flow columns: 96 ['V00', 'V01', 'V02', 'V03', 'V04'] ... ['V91', 'V92', 'V93', 'V94', 'V95']
After cleaning: (4192, 106)
Saved cleaned CSV -> artifacts_person1/cleaned_flows.csv


## 3) Build sequences (X, y)

We turn each row (one SCATS site on one date) into a length‑96 time series.

Then we create sliding windows:
- Input `X[t]` = previous `seq_len` 15‑minute volumes
- Target `y[t]` = the next 15‑minute volume

To help the model learn daily periodicity, we also add **time-of-day** features per step: `sin(2πt/96)` and `cos(2πt/96)`.

In [8]:
seq_len = 12  # 12*15min = 3 hours history

# Split by SCATS site to reduce leakage
site_groups = df['SCATS Number'].astype(str)

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=SEED)
train_idx, test_idx = next(gss.split(df, groups=site_groups))
df_train = df.iloc[train_idx].copy()
df_test = df.iloc[test_idx].copy()

gss2 = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=SEED)  # 0.25 of train -> val => val ~0.2 overall
tr_idx, val_idx = next(gss2.split(df_train, groups=df_train['SCATS Number'].astype(str)))
df_tr = df_train.iloc[tr_idx].copy()
df_val = df_train.iloc[val_idx].copy()

print('Split sizes:', len(df_tr), len(df_val), len(df_test))
print('Unique sites:', df['SCATS Number'].nunique(), 'train/val/test:', df_tr['SCATS Number'].nunique(), df_val['SCATS Number'].nunique(), df_test['SCATS Number'].nunique())

# Time-of-day features for 96 steps
T = 96
tod = np.arange(T)
tod_sin = np.sin(2 * np.pi * tod / T).astype(np.float32)
tod_cos = np.cos(2 * np.pi * tod / T).astype(np.float32)

# Scale flow values (fit on train only)
flow_scaler = StandardScaler()
flow_scaler.fit(df_tr[flow_cols].to_numpy().reshape(-1, 1))

def build_sequences_from_df(d: pd.DataFrame, seq_len: int):
    X_list = []
    y_list = []
    meta = []  # (site, date, step)

    flows = d[flow_cols].to_numpy(dtype=np.float32)  # shape (n_rows, 96)
    flows_scaled = flow_scaler.transform(flows.reshape(-1, 1)).reshape(flows.shape).astype(np.float32)

    sites = d['SCATS Number'].to_numpy()
    dates = d['Date'].to_numpy()

    for i in range(flows_scaled.shape[0]):
        series = flows_scaled[i]
        for t in range(0, T - seq_len):
            win = series[t:t + seq_len]
            # features per step: [flow, sin, cos]
            step_idx = np.arange(t, t + seq_len)
            X = np.stack([win, tod_sin[step_idx], tod_cos[step_idx]], axis=-1)
            y = series[t + seq_len]
            X_list.append(X)
            y_list.append(y)
            meta.append((int(sites[i]), pd.to_datetime(dates[i]), int(t + seq_len)))

    return np.asarray(X_list, dtype=np.float32), np.asarray(y_list, dtype=np.float32), meta

X_tr, y_tr, meta_tr = build_sequences_from_df(df_tr, seq_len)
X_val, y_val, meta_val = build_sequences_from_df(df_val, seq_len)
X_test, y_test, meta_test = build_sequences_from_df(df_test, seq_len)

print('X_tr:', X_tr.shape, 'y_tr:', y_tr.shape)
print('X_val:', X_val.shape, 'y_val:', y_val.shape)
print('X_test:', X_test.shape, 'y_test:', y_test.shape)

Split sizes: 2520 896 776
Unique sites: 40 train/val/test: 24 8 8
X_tr: (211680, 12, 3) y_tr: (211680,)
X_val: (75264, 12, 3) y_val: (75264,)
X_test: (65184, 12, 3) y_test: (65184,)


In [9]:
# Optional: export the row-level train/val/test split as CSV (useful when your team expects 1 train + 1 test file)
os.makedirs('artifacts_person1', exist_ok=True)

base_cols = [c for c in ['SCATS Number', 'Date', 'NB_LATITUDE', 'NB_LONGITUDE'] if c in df.columns]
cols_to_save = base_cols + flow_cols

df_tr[cols_to_save].to_csv('artifacts_person1/train_rows.csv', index=False)
df_val[cols_to_save].to_csv('artifacts_person1/val_rows.csv', index=False)
df_test[cols_to_save].to_csv('artifacts_person1/test_rows.csv', index=False)

print('Saved row-level splits:')
print('-', 'artifacts_person1/train_rows.csv', len(df_tr))
print('-', 'artifacts_person1/val_rows.csv', len(df_val))
print('-', 'artifacts_person1/test_rows.csv', len(df_test))

Saved row-level splits:
- artifacts_person1/train_rows.csv 2520
- artifacts_person1/val_rows.csv 896
- artifacts_person1/test_rows.csv 776


In [10]:
# Helper to convert between scaled and original flow units (vehicles / 15 minutes)

def inverse_flow_scale(x_scaled: np.ndarray) -> np.ndarray:
    x_scaled = np.asarray(x_scaled).reshape(-1, 1)
    return flow_scaler.inverse_transform(x_scaled).reshape(-1)

# For evaluation in original units
y_test_true = inverse_flow_scale(y_test)
print('Example y_test (scaled -> true):')
for i in range(5):
    print(float(y_test[i]), '->', float(y_test_true[i]))

Example y_test (scaled -> true):
-1.0235106945037842 -> 15.999992370605469
-1.0235106945037842 -> 15.999992370605469
-1.0578359365463257 -> 13.0
-1.0235106945037842 -> 15.999992370605469
-1.0235106945037842 -> 15.999992370605469


## 4) Person 1 — LSTM model

In [11]:
class LSTMRegressor(nn.Module):
    def __init__(self, n_features: int, hidden_size: int = 64, dense_size: int = 32):
        super().__init__()
        self.lstm = nn.LSTM(input_size=n_features, hidden_size=hidden_size, num_layers=1, batch_first=True)
        self.fc1 = nn.Linear(hidden_size, dense_size)
        self.fc2 = nn.Linear(dense_size, 1)
        self.relu = nn.ReLU()

    def forward(self, x):
        # x: (batch, seq_len, n_features)
        out, _ = self.lstm(x)
        last = out[:, -1, :]
        x = self.relu(self.fc1(last))
        x = self.fc2(x)
        return x.squeeze(-1)

lstm = LSTMRegressor(n_features=X_tr.shape[-1]).to(device)
print(lstm)

LSTMRegressor(
  (lstm): LSTM(3, 64, batch_first=True)
  (fc1): Linear(in_features=64, out_features=32, bias=True)
  (fc2): Linear(in_features=32, out_features=1, bias=True)
  (relu): ReLU()
)


In [12]:
# Training loop (PyTorch)
batch_size = 64
epochs = 60
patience = 8  # early stopping
min_delta = 1e-6

train_ds = TensorDataset(torch.from_numpy(X_tr), torch.from_numpy(y_tr))
val_ds = TensorDataset(torch.from_numpy(X_val), torch.from_numpy(y_val))

train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, drop_last=False)
val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False, drop_last=False)

criterion = nn.MSELoss()
optimizer = torch.optim.Adam(lstm.parameters(), lr=1e-3)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=4, min_lr=1e-6
 )

best_val = float('inf')
best_state = None
wait = 0
history_lstm = []

for epoch in range(1, epochs + 1):
    lstm.train()
    train_losses = []
    for xb, yb in train_loader:
        xb = xb.to(device)
        yb = yb.to(device)
        optimizer.zero_grad(set_to_none=True)
        pred = lstm(xb)
        loss = criterion(pred, yb)
        loss.backward()
        optimizer.step()
        train_losses.append(loss.detach().item())

    lstm.eval()
    val_losses = []
    with torch.no_grad():
        for xb, yb in val_loader:
            xb = xb.to(device)
            yb = yb.to(device)
            pred = lstm(xb)
            loss = criterion(pred, yb)
            val_losses.append(loss.detach().item())

    train_loss = float(np.mean(train_losses))
    val_loss = float(np.mean(val_losses))
    scheduler.step(val_loss)
    lr = float(optimizer.param_groups[0]['lr'])
    history_lstm.append({'epoch': epoch, 'train_loss': train_loss, 'val_loss': val_loss, 'lr': lr})
    print(f"Epoch {epoch:03d} | train_loss={train_loss:.5f} | val_loss={val_loss:.5f} | lr={lr:.2e}")

    if val_loss < (best_val - min_delta):
        best_val = val_loss
        best_state = {k: v.detach().cpu().clone() for k, v in lstm.state_dict().items()}
        wait = 0
    else:
        wait += 1
        if wait >= patience:
            print(f"Early stopping at epoch {epoch} (best val_loss={best_val:.5f})")
            break

if best_state is not None:
    lstm.load_state_dict(best_state)

Epoch 001 | train_loss=0.07526 | val_loss=0.04530 | lr=1.00e-03
Epoch 002 | train_loss=0.06108 | val_loss=0.04162 | lr=1.00e-03
Epoch 003 | train_loss=0.05954 | val_loss=0.04303 | lr=1.00e-03
Epoch 004 | train_loss=0.05838 | val_loss=0.04064 | lr=1.00e-03
Epoch 005 | train_loss=0.05768 | val_loss=0.04057 | lr=1.00e-03
Epoch 006 | train_loss=0.05709 | val_loss=0.04295 | lr=1.00e-03
Epoch 007 | train_loss=0.05662 | val_loss=0.04071 | lr=1.00e-03
Epoch 008 | train_loss=0.05600 | val_loss=0.04370 | lr=1.00e-03
Epoch 009 | train_loss=0.05557 | val_loss=0.04088 | lr=1.00e-03
Epoch 010 | train_loss=0.05506 | val_loss=0.04002 | lr=1.00e-03
Epoch 011 | train_loss=0.05473 | val_loss=0.04065 | lr=1.00e-03
Epoch 012 | train_loss=0.05435 | val_loss=0.04148 | lr=1.00e-03
Epoch 013 | train_loss=0.05403 | val_loss=0.04107 | lr=1.00e-03
Epoch 014 | train_loss=0.05361 | val_loss=0.04091 | lr=1.00e-03
Epoch 015 | train_loss=0.05323 | val_loss=0.04027 | lr=5.00e-04
Epoch 016 | train_loss=0.05186 | val_los

In [13]:
def regression_metrics(y_true, y_pred):
    y_true = np.asarray(y_true).reshape(-1)
    y_pred = np.asarray(y_pred).reshape(-1)
    mae = mean_absolute_error(y_true, y_pred)
    mse = mean_squared_error(y_true, y_pred)
    rmse = float(math.sqrt(mse))
    r2 = r2_score(y_true, y_pred)
    return {'MAE': float(mae), 'RMSE': float(rmse), 'R2': float(r2)}

lstm.eval()
with torch.no_grad():
    pred_lstm_scaled = lstm(torch.from_numpy(X_test).to(device)).detach().cpu().numpy().reshape(-1)

pred_lstm_true = inverse_flow_scale(pred_lstm_scaled)

metrics_lstm = regression_metrics(y_test_true, pred_lstm_true)
metrics_lstm

{'MAE': 13.85396671295166,
 'RMSE': 22.173574800505076,
 'R2': 0.9394356608390808}

In [14]:
# Predictions + metrics (vehicles per 15 minutes, original units)

lstm.eval()
with torch.no_grad():
    pred_lstm_scaled = lstm(torch.from_numpy(X_test).to(device)).detach().cpu().numpy().reshape(-1)

pred_lstm_veh_15min = inverse_flow_scale(pred_lstm_scaled)
y_true_veh_15min = y_test_true

metrics_lstm = regression_metrics(y_true_veh_15min, pred_lstm_veh_15min)

results = pd.DataFrame([
    {'model': 'LSTM_torch', **metrics_lstm},
])

pred_df = pd.DataFrame({
    'scats_site': [m[0] for m in meta_test],
    'date': [m[1] for m in meta_test],
    'step_index_0to95': [m[2] for m in meta_test],
    'y_true_veh_15min': y_true_veh_15min,
    'pred_lstm_veh_15min': pred_lstm_veh_15min,
})

display(results)
pred_df.head()

,model,MAE,RMSE,R2
0,LSTM_torch,13.853967,22.173575,0.939436


,scats_site,date,step_index_0to95,y_true_veh_15min,pred_lstm_veh_15min
0,2827,2006-10-01 00:15:00,12,15.999992,16.792801
1,2827,2006-10-01 00:15:00,13,15.999992,17.286026
2,2827,2006-10-01 00:15:00,14,13.000000,16.276070
3,2827,2006-10-01 00:15:00,15,15.999992,14.312531
4,2827,2006-10-01 00:15:00,16,15.999992,14.484543


## 4) Person 1 outputs (save model + predictions)

This writes everything needed for Person 1 into `artifacts_person1/`:
- trained LSTM model
- scaler used for flow normalization
- test predictions + metrics

In [15]:
os.makedirs('artifacts_person1', exist_ok=True)

# Save LSTM (PyTorch) model weights + minimal config
torch.save(lstm.state_dict(), 'artifacts_person1/lstm_flow_15min_torch.pt')
with open('artifacts_person1/lstm_flow_15min_torch_config.json', 'w', encoding='utf-8') as f:
    json.dump({
        'seq_len': int(seq_len),
        'n_features': int(X_tr.shape[-1]),
        'hidden_size': 64,
        'dense_size': 32,
        'backend': 'pytorch',
    }, f, ensure_ascii=False, indent=2)

# Save scaler (needed to convert model outputs back to vehicles/15min)
joblib.dump(flow_scaler, 'artifacts_person1/flow_scaler.joblib')

# Save results
pred_df.to_csv('artifacts_person1/predictions_test.csv', index=False)
results.to_csv('artifacts_person1/model_results.csv', index=False)

print('Saved Person 1 artifacts to artifacts_person1/')

Saved Person 1 artifacts to artifacts_person1/


## Ghi chú bàn giao (để người sau làm tiếp)

Person 1 đã tạo ra:
- `artifacts_person1/lstm_flow_15min_torch.pt` + `lstm_flow_15min_torch_config.json` (model PyTorch)
- `artifacts_person1/flow_scaler.joblib` (chuẩn hoá/giải chuẩn hoá)
- `artifacts_person1/predictions_test.csv` + `model_results.csv`

Người tiếp theo nên làm:
- Nếu làm Person 2: dùng **đúng Cell 9** (sequence builder) để tạo X,y y hệt, rồi thay model bằng GRU / model khác để so sánh.
- Nếu làm Person 3: dùng công thức travel-time từ `Traffic Flow to Travel Time Conversion v1.0.pdf` + delay 30s từ spec để đổi predicted flow → speed → time và chạy top‑k routes trên graph Part A.